# LLM Zoomcamp 2026 — Homework 4: Evaluation

This notebook follows the official **Homework: Evaluation** instructions.

It evaluates text, vector, and hybrid search over the 72 course lesson pages pinned to commit `8c1834d`.

> Run the notebook from the Homework 4 project directory. The project must reuse `embedder.py`, the downloaded ONNX model, and the search setup from Homework 2.

In [1]:
import json
import os
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv

from evaluation_utils import llm_structured

load_dotenv()
client = OpenAI()


class Questions(BaseModel):
    questions: list[str]

## Load the 72 lesson pages

The repository is pinned to commit `8c1834d`, as required by the homework.

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print("Number of lesson pages:", len(documents))
assert len(documents) == 72

Number of lesson pages: 72


In [3]:
documents[0].keys()

dict_keys(['content', 'filename'])

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [5]:
first_three_filenames = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

first_three_documents = [
    document
    for document in documents
    if document["filename"] in first_three_filenames
]

first_three_documents = sorted(
    first_three_documents,
    key=lambda document: first_three_filenames.index(
        document["filename"]
    ),
)

assert len(first_three_documents) == 3

[document["filename"] for document in first_three_documents]

['01-agentic-rag/lessons/01-intro.md',
 '01-agentic-rag/lessons/02-environment.md',
 '01-agentic-rag/lessons/03-rag.md']

In [6]:
generated_questions = []
usages = []

for document in first_three_documents:
    user_prompt = json.dumps(
        {
            "filename": document["filename"],
            "content": document["content"],
        }
    )

    questions, usage = llm_structured(
        client=client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
        model="gpt-5.4-mini",
    )

    generated_questions.append(
        {
            "filename": document["filename"],
            "questions": questions.questions,
        }
    )

    usages.append(usage)

### Q1 : When generating questions for the first 3 lesson pages, what is the average number of input tokens across these 3 calls?

In [7]:
input_tokens = [
    usage.input_tokens
    for usage in usages
]

average_input_tokens = sum(input_tokens) / len(input_tokens)

print("Input tokens per call:", input_tokens)
print("Average input tokens:", average_input_tokens)

Input tokens per call: [1021, 1287, 1754]
Average input tokens: 1354.0


## Load the provided full ground truth



In [8]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

!curl -L {PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv \
  -o ground-truth.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 48627  100 48627    0     0   674k      0 --:--:-- --:--:-- --:--:--  678k


In [9]:
import pandas as pd

ground_truth_df = pd.read_csv("ground-truth.csv")

ground_truth = ground_truth_df.to_dict(
    orient="records"
)

print("Number of questions:", len(ground_truth))
print("Columns:", ground_truth_df.columns.tolist())

ground_truth_df.head()

Number of questions: 360
Columns: ['question', 'filename']


,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md


In [10]:
ground_truth = ground_truth_df.to_dict(orient="records")

In [11]:
ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

## Create chunks 

The required configuration is `size=2000` and `step=1000`. It should produce 295 chunks.

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

### Text Search

In [13]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

text_index.fit(chunks)

In [14]:
def text_search(query, num_results=5):
    return text_index.search(
        query=query,
        num_results=num_results,
    )

### Vector Search

In [15]:
from embedder import Embedder
from minsearch import VectorSearch

In [16]:
embedder = Embedder()

embeddings = embedder.encode_batch(
    [chunk["content"] for chunk in chunks]
)

In [17]:
vector_index = VectorSearch(
    keyword_fields=["filename"]
)

vector_index.fit(
    embeddings,
    chunks
)

In [18]:
def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)

    return vector_index.search(
        query_vector,
        num_results=num_results,
    )

## Q2: After running text_search for the first ground truth question, what is the filename of the first result?

In [19]:
q = ground_truth[0]["question"]
text_results = text_search(q)
q2_answer = text_results[0]["filename"]

print("First ground-truth question:", q)
print("Q2 answer:", q2_answer)

First ground-truth question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
Q2 answer: 01-agentic-rag/lessons/03-rag.md


## Q3: After running vector_search for the same question, what is the filename of the first result?

In [20]:
vector_results = vector_search(q)

q3_answer = vector_results[0]["filename"]

print("Q3 answer:", q3_answer)

Q3 answer: 01-agentic-rag/lessons/01-intro.md


## Evaluation metrics

A returned chunk is relevant when its `filename` matches the ground-truth `filename`. Hit Rate checks whether the correct page appears anywhere in the returned list. MRR also rewards a higher rank.

In [21]:
def compute_relevance(search_function, ground_truth):
    relevance_total = []

    for record in ground_truth:
        results = search_function(
            record["question"]
        )

        relevance = [
            result["filename"] == record["filename"]
            for result in results
        ]

        relevance_total.append(relevance)

    return relevance_total

### Hit Rate

In [31]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

### MRR

In [32]:
def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] is True:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance_total)

### Evaluate

In [35]:
def evaluate(search_function, ground_truth):
    relevance_total = compute_relevance(search_function, ground_truth)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }
    

##  Q4: After evaluating text_search on the ground truth, what is the Hit Rate

In [36]:
text_search_metrics = evaluate(text_search, ground_truth)

print(text_search_metrics)

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}


## Q5: After evaluating vector_search on the ground truth, what is the MRR?

In [37]:
vector_search_metrics = evaluate(
    vector_search,
    ground_truth
)

q5_value = vector_search_metrics["mrr"]

print("Vector search metrics:", vector_search_metrics)
print("Q5 MRR:", q5_value)

Vector search metrics: {'hit_rate': 0.725, 'mrr': 0.5486111111111112}
Q5 MRR: 0.5486111111111112


## Tune hybrid search

The RRF implementation and `hybrid_search` definition below are the ones provided in the homework.

In [38]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [39]:
hybrid_results = {}

for k in [1, 50, 100, 200]:
    search_function = lambda query, k=k: hybrid_search(query, k=k)
    hybrid_results[k] = evaluate(search_function, ground_truth)

hybrid_results

{1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449},
 50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667},
 100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667},
 200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}}

In [40]:
best_mrr = max(result["mrr"] for result in hybrid_results.values())
q6_answer = min(
    k
    for k, result in hybrid_results.items()
    if result["mrr"] == best_mrr
)

print("Q6 best k:", q6_answer)
print("Best MRR:", best_mrr)

Q6 best k: 1
Best MRR: 0.6481944444444449
